# DeQA Label Analysis

This notebook analyzes pseudo-labels generated by the DeQA labeling infrastructure.

## Setup

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

## Load Labels

In [ ]:
def load_labels(path: str) -> pd.DataFrame:
    """Load JSONL labels into DataFrame."""
    records = []
    with open(path) as f:
        for line in f:
            data = json.loads(line.strip())
            record = {
                'image': data['image'],
                'dataset': data.get('dataset', 'unknown'),
                'mode': data.get('mode', 'unknown'),
            }
            scores = data.get('scores', {})
            for dim in ['overall', 'sharpness', 'color']:
                record[dim] = scores.get(dim, np.nan)
            records.append(record)
    return pd.DataFrame(records)

# Load label files
labels_dir = Path('../results/deqa_labels')

specialist_df = load_labels(labels_dir / 'diqa-5000_specialist_labels.jsonl')
ensemble_df = load_labels(labels_dir / 'diqa-5000_ensemble_labels.jsonl')
ocr_quality_df = load_labels(labels_dir / 'ocr-quality_specialist_labels.jsonl')

print(f"Specialist labels: {len(specialist_df)}")
print(f"Ensemble labels: {len(ensemble_df)}")
print(f"OCR-Quality labels: {len(ocr_quality_df)}")

## Score Distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

dimensions = ['overall', 'sharpness', 'color']
colors = ['#2ecc71', '#3498db', '#e74c3c']

for ax, dim, color in zip(axes, dimensions, colors):
    sns.histplot(specialist_df[dim], bins=30, kde=True, ax=ax, color=color, alpha=0.7)
    ax.axvline(specialist_df[dim].mean(), color='black', linestyle='--', label=f'Mean: {specialist_df[dim].mean():.2f}')
    ax.set_title(f'{dim.capitalize()} Score Distribution')
    ax.set_xlabel('Score')
    ax.set_xlim(1, 5)
    ax.legend()

plt.suptitle('DIQA-5000 Score Distributions (Specialist Mode)', fontsize=14)
plt.tight_layout()
plt.show()

## Specialist vs Ensemble Comparison

In [ ]:
# Merge on image path
comparison_df = specialist_df.merge(
    ensemble_df[['image', 'overall']],
    on='image',
    suffixes=('_specialist', '_ensemble')
)

# Scatter plot
fig, ax = plt.subplots(figsize=(8, 8))

ax.scatter(
    comparison_df['overall_specialist'],
    comparison_df['overall_ensemble'],
    alpha=0.3,
    s=10
)

# Perfect correlation line
ax.plot([1, 5], [1, 5], 'r--', label='Perfect correlation')

# Calculate correlation
srcc, _ = stats.spearmanr(comparison_df['overall_specialist'], comparison_df['overall_ensemble'])
plcc, _ = stats.pearsonr(comparison_df['overall_specialist'], comparison_df['overall_ensemble'])

ax.set_xlabel('Specialist Overall Score')
ax.set_ylabel('Ensemble Overall Score')
ax.set_title(f'Specialist vs Ensemble Comparison\nSRCC={srcc:.4f}, PLCC={plcc:.4f}')
ax.legend()
ax.set_xlim(1, 5)
ax.set_ylim(1, 5)
ax.set_aspect('equal')

plt.tight_layout()
plt.show()

## Dimension Correlations

In [ ]:
# Correlation matrix for specialist dimensions
dims = ['overall', 'sharpness', 'color']
corr_matrix = specialist_df[dims].corr(method='spearman')

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt='.4f',
    cmap='coolwarm',
    center=0,
    ax=ax,
    vmin=0.8,
    vmax=1.0
)
ax.set_title('Dimension Correlation Matrix (Spearman)')
plt.tight_layout()
plt.show()

## Dataset Comparison

In [ ]:
# Compare DIQA-5000 vs OCR-Quality
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, dim in zip(axes, dimensions):
    # DIQA-5000
    sns.kdeplot(specialist_df[dim], ax=ax, label='DIQA-5000', color='blue')
    # OCR-Quality
    sns.kdeplot(ocr_quality_df[dim], ax=ax, label='OCR-Quality', color='orange')
    
    ax.set_title(f'{dim.capitalize()} Score Distribution')
    ax.set_xlabel('Score')
    ax.set_xlim(1, 5)
    ax.legend()

plt.suptitle('Score Distribution by Dataset', fontsize=14)
plt.tight_layout()
plt.show()

## Summary Statistics

In [ ]:
def get_summary_stats(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Get summary statistics for a label set."""
    stats_dict = {'dataset': name}
    for dim in dimensions:
        if dim in df.columns and not df[dim].isna().all():
            stats_dict[f'{dim}_mean'] = df[dim].mean()
            stats_dict[f'{dim}_std'] = df[dim].std()
            stats_dict[f'{dim}_min'] = df[dim].min()
            stats_dict[f'{dim}_max'] = df[dim].max()
    return pd.DataFrame([stats_dict])

summary_df = pd.concat([
    get_summary_stats(specialist_df, 'DIQA-5000 (Specialist)'),
    get_summary_stats(ensemble_df, 'DIQA-5000 (Ensemble)'),
    get_summary_stats(ocr_quality_df, 'OCR-Quality (Specialist)'),
])

summary_df.round(3)

## Quality Category Distribution

In [ ]:
def categorize_score(score: float) -> str:
    """Categorize score into quality bins."""
    if score < 1.5:
        return 'Bad (1.0-1.5)'
    elif score < 2.5:
        return 'Poor (1.5-2.5)'
    elif score < 3.5:
        return 'Fair (2.5-3.5)'
    elif score < 4.5:
        return 'Good (3.5-4.5)'
    else:
        return 'Excellent (4.5-5.0)'

specialist_df['quality_category'] = specialist_df['overall'].apply(categorize_score)

category_counts = specialist_df['quality_category'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(10, 6))
category_counts.plot(kind='bar', ax=ax, color=['#e74c3c', '#f39c12', '#f1c40f', '#2ecc71', '#27ae60'])
ax.set_title('DIQA-5000 Quality Category Distribution')
ax.set_xlabel('Quality Category')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)

# Add percentage labels
total = len(specialist_df)
for i, (cat, count) in enumerate(category_counts.items()):
    ax.text(i, count + 50, f'{count/total*100:.1f}%', ha='center')

plt.tight_layout()
plt.show()

## Score Error Analysis

In [ ]:
# Calculate differences between specialist and ensemble
comparison_df['diff'] = comparison_df['overall_specialist'] - comparison_df['overall_ensemble']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram of differences
axes[0].hist(comparison_df['diff'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(0, color='red', linestyle='--', label='Zero difference')
axes[0].set_title('Score Difference Distribution (Specialist - Ensemble)')
axes[0].set_xlabel('Difference')
axes[0].set_ylabel('Count')
axes[0].legend()

# Stats
mae = np.mean(np.abs(comparison_df['diff']))
rmse = np.sqrt(np.mean(comparison_df['diff']**2))

axes[1].text(0.5, 0.6, f'Mean Absolute Error: {mae:.6f}', ha='center', fontsize=14, transform=axes[1].transAxes)
axes[1].text(0.5, 0.4, f'RMSE: {rmse:.6f}', ha='center', fontsize=14, transform=axes[1].transAxes)
axes[1].axis('off')
axes[1].set_title('Error Metrics')

plt.tight_layout()
plt.show()

print(f"\nStatistics:")
print(f"  Mean difference: {comparison_df['diff'].mean():.6f}")
print(f"  Std difference: {comparison_df['diff'].std():.6f}")
print(f"  Max difference: {comparison_df['diff'].abs().max():.6f}")